# Data Gathering

*I am in the process of adapting this old assignment into my outline for webscraping my data for my final proj*

I will be grabbing the discographies of
- joni mitchell
- indigo girls
- natalie merchant?
- melissa etheridge?

## Import packages

In [1]:
import numpy as np
import pandas as pd
import requests
import time
import json
from bs4 import BeautifulSoup

## Get Data

In [ ]:
def get_album_urls(discography_url, headers, base_url='https://jonimitchell.com'):
    """
    Scrapes discography index and returns a list of dictionaries,
    each containing the album's URL, title, and release date.
    """

    # get discography page
    print(f"Pinging {discography_url} for album index...")
    r = requests.get(discography_url, headers=headers)

    # check if request was successful
    if r.status_code != 200:
        print(f"Warning: Recieved status code {r.status_code}.")
        return []

    # parse the html with BeautifulSoup
    soup = BeautifulSoup(r.text, 'html.parser')

    # target the div that contains all the info for a single album
    gallery_items = soup.find_all('div', class_='gallery-item')

    albums = []

    for item in gallery_items:
        # extract the URL
        a_tag = item.find('a', href=True)
        if not a_tag or 'album.cfm' not in a_tag['href']:
            continue # skip item if it doesn't have a valid album link
        
        full_url = base_url + a_tag['href']

        # extract title
        title_tag = item.find('span', class_='albumtitle')
        title = title_tag.text.strip() if title_tag else 'Unknown Title'
        
        # extract release date
        date_tag = item.find('span', class_='released')
        if date_tag:
            # remove the word "Released' so you just have Month Day Year
            release_date = date_tag.text.replace('Released ', '').strip()
        else:
            release_date = 'Unknown Date'

        # bind the info together into a single record
        album_record = {
            'album_url': full_url,
            'album_title': title,
            'release_date': release_date
        }

        albums.append(album_record)

    return albums

In [ ]:
def get_album_tracks(album_record, headers, base_url='https://jonimitchell.com'):
    """
    Takes a single album dictionary, visits its page, and returns a list of
    dictionaries representing the individual songs on that album.
    """

    album_url = album_record['album_url']
    print(f"  -> Digging into album: {album_record['album_title']}...")

    r = requests.get(album_url, headers=headers)
    time.sleep(2) # be polite to server and wait a few seconds before the next request
    
    if r.status_code != 200:
        print(f"  Warning: Recieved status code {r.status_code} for album page.")
        return []

    soup = BeautifulSoup(r.text, 'html.parser')

    # find the div that contains the track listing
    tracklist_ul = soup.find('ul', class_='tracklist') # ul in html means unordered list

    if not tracklist_ul:
        print("  Warning: No tracklist found on this page.")
        return []

    # extract links within the tracklist
    song_links = tracklist_ul.find_all('a', href=True)

    songs = []

    for link in song_links:
        href = link['href']

        # make sure its a song link
        if 'song.cfm?id=' not in href:
            continue # skip if it's not a valid song link

        # extract songs within tracklist
        song_title = link.contents[0].strip()

        # make sure the song title is not empty
        if not song_title:
            continue # skip if song title is empty

        # get url and title for each song
        song_record = {
            'song_url': base_url + href,
            'song_title': song_title,

In [ ]:
def get_song_lyrics(song_record, headers):
    """
    Visits a specific song page, extracts the author, lyrics, and copyright,
    and returns a complete, finalized dictionary for the corpus.
    """

    song_url = song_record['song_url']
    print(f"    -> Fetching lyrics for song: {song_record['song_title']}")

    r = requests.get(song_url, headers=headers)
    time.sleep(2) # be polite to server and wait a few seconds

    if r.status_code != 200:
        print(f"    Warning: Recieved status code {r.status_code} for song page.")
        song_record['author'] = None
        song_record['copyright'] = None
        song_record['lyrics'] = None
        return song_record

    soup = BeautifulSoup(r.text, 'html.parser')

    # extract the author
    # note the author tag is outside of the lyrics div, so we have to look for it separately by searching the whole soup
    author_tag = soup.find('p', class_='author')
    if author_tag:
        author_text = author_tag.text.strip()
        # clean the data: remove "by" if it exists at start of string and strip whitespace
        if author_text.lower().startswith('by '):
            author_text = author_text[3:].strip()
        song_record['author'] = author_text
    else:
        song_record['author'] = "Unknown"

    # target the lyrics container
    lyrics_div = soup.find('div', class_='songlyrics')

    if not lyrics_div:
        print("    Warning: No lyrics found on this page.")
        song_record['copyright'] = None
        song_record['lyrics'] = None
        return song_record

    # extract copyright metadata
    copyright_tag = lyrics_div.find('p', class_='lyricscopyright')
    if copyright_tag:
        song_record['copyright'] = copyright_tag.text.replace('© ', '').strip()
    else:
        song_record['copyright'] = "Unknown"

    # extract OHCO-ready lyrics text
    lyrics_paragraphs = lyrics_div.find_all('p', class_=lambda x: x != 'lyricscopyright') # get all paragraphs except the copyright one
    
    raw_lyrics = []
    for p in lyrics_paragraphs:
        # get_text(separator='\n') will replace all <br> tags with actual line breaks
        raw_lyrics.append(p.get_text(separator='\n').strip())
    
    song_record['lyrics'] = '\n\n'.join(raw_lyrics)

    return song_record

In [19]:
# setup
base_url = 'https://jonimitchell.com'
discography_url = f'{base_url}/music/index.cfm'
useragent = f'BeanBot/2.2 (bhj3vc@virginia.edu) python-requests/{requests.__version__}'
headers = {'User-Agent': useragent,
        'From': 'bhj3vc@virginia.edu'}

# execute level 1
albums = get_album_urls(discography_url, headers)
print(f"\nSuccess! Loaded {len(albums)} albums into the pipeline.")

Pinging https://jonimitchell.com/music/index.cfm for album index...

Success! Loaded 50 albums into the pipeline.


In [20]:
albums

[{'album_url': 'https://jonimitchell.com/music/album.cfm?id=2',
  'album_title': 'Song to a Seagull',
  'release_date': 'March 1968'},
 {'album_url': 'https://jonimitchell.com/music/album.cfm?id=3',
  'album_title': 'Clouds',
  'release_date': 'May 1969'},
 {'album_url': 'https://jonimitchell.com/music/album.cfm?id=4',
  'album_title': 'Ladies of the Canyon',
  'release_date': 'April 6, 1970'},
 {'album_url': 'https://jonimitchell.com/music/album.cfm?id=35',
  'album_title': 'Amchitka',
  'release_date': 'November 2009'},
 {'album_url': 'https://jonimitchell.com/music/album.cfm?id=5',
  'album_title': 'Blue',
  'release_date': 'June 22, 1971'},
 {'album_url': 'https://jonimitchell.com/music/album.cfm?id=34',
  'album_title': 'The World of Joni Mitchell',
  'release_date': '1971'},
 {'album_url': 'https://jonimitchell.com/music/album.cfm?id=6',
  'album_title': 'For the Roses',
  'release_date': 'November 21, 1972'},
 {'album_url': 'https://jonimitchell.com/music/album.cfm?id=7',
  'alb

In [ ]:
def build_joni_corpus(discography_url):
    all_corpus_data = [] # will hold all final rows

    # set headers for polite scraping
    useragent = f'BeanBot/2.2 (bhj3vc@virginia.edu) python-requests/{requests.__version__}'
    headers = {'User-Agent': useragent,
            'From': 'bhj3vc@virginia.edu'}

    # --- Level 1: get album urls ---
    # get discography page
    print(f"Pinging {discography_url}...")
    r = requests.get(discography_url, headers=headers)

    # parse the html with BeautifulSoup
    soup = BeautifulSoup(r.text, 'html.parser')

    # find all links on the page
    all_links = soup.find_all('a', href=True)

    # extract and format the album links
    album_urls = []
    for link in all_links:
        href = link['href']
        
        # only want links that point to an album page
        if 'album.cfm?id=' in href:
            # the href is a relative link, so we need to add the base url to get the full url
            full_url = base_url + href
            album_urls.append(full_url)

    # clean up the album urls (remove duplicates and convert back to a list)
    album_urls = list(set(album_urls))

    print(f"\nSuccess! Found {len(album_urls)} unique album URLs.")
    print("First 5 links to verify:")
    for url in album_urls[:5]:
        print(url)

    # --- Level 2: get song urls by iterating through albums ---
    TODO

    

### Part b
Extract all 20 of the book titles and save them in a list. [8 points]

In [17]:
titletaglist = booksoup.find_all('a', title=True)
titletaglist

[<a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a>,
 <a href="catalogue/tipping-the-velvet_999/index.html" title="Tipping the Velvet">Tipping the Velvet</a>,
 <a href="catalogue/soumission_998/index.html" title="Soumission">Soumission</a>,
 <a href="catalogue/sharp-objects_997/index.html" title="Sharp Objects">Sharp Objects</a>,
 <a href="catalogue/sapiens-a-brief-history-of-humankind_996/index.html" title="Sapiens: A Brief History of Humankind">Sapiens: A Brief History ...</a>,
 <a href="catalogue/the-requiem-red_995/index.html" title="The Requiem Red">The Requiem Red</a>,
 <a href="catalogue/the-dirty-little-secrets-of-getting-your-dream-job_994/index.html" title="The Dirty Little Secrets of Getting Your Dream Job">The Dirty Little Secrets ...</a>,
 <a href="catalogue/the-coming-woman-a-novel-based-on-the-life-of-the-infamous-feminist-victoria-woodhull_993/index.html" title="The Coming Woman: A Novel Based on the Life of the 

In [18]:
# titles = [] # initialize an empty list to store the book titles

# # for book in titletaglist:
# #     #print(book.get('title'))
# #     titles.append(book.get('title'))
    
# # print(titles)

titles = [book.get('title') for book in titletaglist]
titles

['A Light in the Attic',
 'Tipping the Velvet',
 'Soumission',
 'Sharp Objects',
 'Sapiens: A Brief History of Humankind',
 'The Requiem Red',
 'The Dirty Little Secrets of Getting Your Dream Job',
 'The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull',
 'The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics',
 'The Black Maria',
 'Starving Hearts (Triangular Trade Trilogy, #1)',
 "Shakespeare's Sonnets",
 'Set Me Free',
 "Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)",
 'Rip it Up and Start Again',
 'Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991',
 'Olio',
 'Mesaerion: The Best Science Fiction Stories 1800-1849',
 'Libertarianism for Beginners',
 "It's Only the Himalayas"]

In [19]:
# indexing and get appear to do the same thing
# titles2 = [book['title'] for book in titletaglist]
# titles2

**Question: Do indexing and get ever not accomplish the same thing?**

### Part c
Extract the price of each of the 20 books and save these prices in a list. (The prices are listed in British pounds, and include the £ symbol. Remove the £ symbols: if you’ve saved the prices in a list named `prices`, then the following code should work: `prices = [s.replace('Â£', '') for s in prices]`.) [8 points]

In [20]:
pricetaglist = booksoup.find_all(attrs={'class':'price_color'})
pricetaglist

# prices = []
# for tag in pricetaglist:
#     #print(tag.string)
#     prices.append(tag.string)


# prices = [tag.string for tag in pricetaglist]
# #prices

# prices = [s.replace('Â£', '') for s in prices]
# prices

# combine into one line
prices = [tag.string.replace('Â£', '') for tag in pricetaglist]
prices

['51.77',
 '53.74',
 '50.10',
 '47.82',
 '54.23',
 '22.65',
 '33.34',
 '17.93',
 '22.60',
 '52.15',
 '13.99',
 '20.66',
 '17.46',
 '52.29',
 '35.02',
 '57.25',
 '23.88',
 '37.59',
 '51.33',
 '45.17']

### Part d
Extract the star level ratings for the 20 books. [Hint: for tags such as `<p class="star-rating One">` in which the class has a space, the class is actually a list in which the first item in the list is `"star-rating"` and the second item in the list is `"One"`. It's possible to search on either item in this list.][8 points]

In [21]:
startaglist = booksoup.find_all('p', attrs={'class':'star-rating'})

#len(startags)

# stars = []
# for x in range(len(startaglist)):
#     # print(startaglist[x]['class'][1])
#     stars.append(startaglist[x]['class'][1]) # get the second item in the list 'class'
# stars

# as a list comprehension
stars = [startaglist[x]['class'][1] for x in range(len(startaglist))]
stars

['Three',
 'One',
 'One',
 'Four',
 'Five',
 'One',
 'Four',
 'Three',
 'Four',
 'One',
 'Two',
 'Four',
 'Five',
 'Five',
 'Five',
 'Three',
 'One',
 'One',
 'Two',
 'Two']

### Part e
Extract the URLs for the JPEG thumbnail images that show the covers of the 20 books. (Maybe we want to mine the images to build models that predict the star level, literally judging books by their covers.) [8 points]

In [22]:
srctaglist = booksoup.find_all('img', src=True)
srctaglist

[<img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>,
 <img alt="Tipping the Velvet" class="thumbnail" src="media/cache/26/0c/260c6ae16bce31c8f8c95daddd9f4a1c.jpg"/>,
 <img alt="Soumission" class="thumbnail" src="media/cache/3e/ef/3eef99c9d9adef34639f510662022830.jpg"/>,
 <img alt="Sharp Objects" class="thumbnail" src="media/cache/32/51/3251cf3a3412f53f339e42cac2134093.jpg"/>,
 <img alt="Sapiens: A Brief History of Humankind" class="thumbnail" src="media/cache/be/a5/bea5697f2534a2f86a3ef27b5a8c12a6.jpg"/>,
 <img alt="The Requiem Red" class="thumbnail" src="media/cache/68/33/68339b4c9bc034267e1da611ab3b34f8.jpg"/>,
 <img alt="The Dirty Little Secrets of Getting Your Dream Job" class="thumbnail" src="media/cache/92/27/92274a95b7c251fea59a2b8a78275ab4.jpg"/>,
 <img alt="The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull" class="thumbnail" src="media/cache/3d/54/3d54940e57e662c4dd1f3ff00c78cc6

In [23]:
srctaglist[1]['src'] # test to see if i've reached the src attribute

# srctaglist[2].get('src')


'media/cache/26/0c/260c6ae16bce31c8f8c95daddd9f4a1c.jpg'

In [24]:
srcs = [srctaglist[x]['src'] for x in range(len(srctaglist))]
srcs

['media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg',
 'media/cache/26/0c/260c6ae16bce31c8f8c95daddd9f4a1c.jpg',
 'media/cache/3e/ef/3eef99c9d9adef34639f510662022830.jpg',
 'media/cache/32/51/3251cf3a3412f53f339e42cac2134093.jpg',
 'media/cache/be/a5/bea5697f2534a2f86a3ef27b5a8c12a6.jpg',
 'media/cache/68/33/68339b4c9bc034267e1da611ab3b34f8.jpg',
 'media/cache/92/27/92274a95b7c251fea59a2b8a78275ab4.jpg',
 'media/cache/3d/54/3d54940e57e662c4dd1f3ff00c78cc64.jpg',
 'media/cache/66/88/66883b91f6804b2323c8369331cb7dd1.jpg',
 'media/cache/58/46/5846057e28022268153beff6d352b06c.jpg',
 'media/cache/be/f4/bef44da28c98f905a3ebec0b87be8530.jpg',
 'media/cache/10/48/1048f63d3b5061cd2f424d20b3f9b666.jpg',
 'media/cache/5b/88/5b88c52633f53cacf162c15f4f823153.jpg',
 'media/cache/94/b1/94b1b8b244bce9677c2f29ccc890d4d2.jpg',
 'media/cache/81/c4/81c4a973364e17d01f217e1188253d5e.jpg',
 'media/cache/54/60/54607fe8945897cdcced0044103b10b6.jpg',
 'media/cache/55/33/553310a7162dfbc2c6d19a84da0df9e1.jpg

In [25]:
imageurls = ['https://books.toscrape.com/' + x['src'] for x in srctaglist]
imageurls

['https://books.toscrape.com/media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg',
 'https://books.toscrape.com/media/cache/26/0c/260c6ae16bce31c8f8c95daddd9f4a1c.jpg',
 'https://books.toscrape.com/media/cache/3e/ef/3eef99c9d9adef34639f510662022830.jpg',
 'https://books.toscrape.com/media/cache/32/51/3251cf3a3412f53f339e42cac2134093.jpg',
 'https://books.toscrape.com/media/cache/be/a5/bea5697f2534a2f86a3ef27b5a8c12a6.jpg',
 'https://books.toscrape.com/media/cache/68/33/68339b4c9bc034267e1da611ab3b34f8.jpg',
 'https://books.toscrape.com/media/cache/92/27/92274a95b7c251fea59a2b8a78275ab4.jpg',
 'https://books.toscrape.com/media/cache/3d/54/3d54940e57e662c4dd1f3ff00c78cc64.jpg',
 'https://books.toscrape.com/media/cache/66/88/66883b91f6804b2323c8369331cb7dd1.jpg',
 'https://books.toscrape.com/media/cache/58/46/5846057e28022268153beff6d352b06c.jpg',
 'https://books.toscrape.com/media/cache/be/f4/bef44da28c98f905a3ebec0b87be8530.jpg',
 'https://books.toscrape.com/media/cache/10/48/1048f63

### Part f
Create a dataframe with one row for each of the 20 books, and the book titles, prices, star ratings, and cover JPEG URLs as the four columns. [8 points]

In [26]:
# to construct a clean data frame, we create a dictionary that combines our lists and passes the dictionary to the pd.DataFrame() function
# our lists are: titles, prices, stars, and imageurllist
mydict = {'Title': titles, 'Price': prices, 'Star Rating': stars, 'Cover Image JPEG URL': imageurls}

books_df = pd.DataFrame(mydict)

books_df

,Title,Price,Star Rating,Cover Image JPEG URL
0,A Light in the Attic,51.77,Three,https://books.toscrape.com/media/cache/2c/da/2...
1,Tipping the Velvet,53.74,One,https://books.toscrape.com/media/cache/26/0c/2...
2,Soumission,50.10,One,https://books.toscrape.com/media/cache/3e/ef/3...
3,Sharp Objects,47.82,Four,https://books.toscrape.com/media/cache/32/51/3...
4,Sapiens: A Brief History of Humankind,54.23,Five,https://books.toscrape.com/media/cache/be/a5/b...
5,The Requiem Red,22.65,One,https://books.toscrape.com/media/cache/68/33/6...
6,The Dirty Little Secrets of Getting Your Dream...,33.34,Four,https://books.toscrape.com/media/cache/92/27/9...
7,The Coming Woman: A Novel Based on the Life of...,17.93,Three,https://books.toscrape.com/media/cache/3d/54/3...
8,The Boys in the Boat: Nine Americans and Their...,22.60,Four,https://books.toscrape.com/media/cache/66/88/6...
9,The Black Maria,52.15,One,https://books.toscrape.com/media/cache/58/46/5...


### Part g
Create a function that takes the URL of the webpage to scrape as an input, applies the code you wrote for parts a through e, and generates the dataframe from part f as the output. [10 points]

In [27]:
def books_spider(url):
    """Performs web scraping for any BooksToScrape page given the page url."""

    # make headers
    useragent = f'BeanBot/2.2 (bhj3vc@virginia.edu) python-requests/{requests.__version__}'
    headers = {'User-Agent': useragent,
            'From': 'bhj3vc@virginia.edu'}

    # get and parse HTML
    r = requests.get(url, headers=headers)
    booksoup = BeautifulSoup(r.text, 'html.parser')
    
    # make taglists
    titletaglist = booksoup.find_all('a', title=True)
    pricetaglist = booksoup.find_all(attrs={'class':'price_color'})
    startaglist = booksoup.find_all('p', attrs={'class':'star-rating'})
    srctaglist = booksoup.find_all('img', src=True)
    
    # pull info into lists
    titles = [book['title'] for book in titletaglist]   #.get would also work here right?
    prices = [tag.string.replace('Â£', '') for tag in pricetaglist]
    stars = [startaglist[x]['class'][1] for x in range(len(startaglist))]
    imageurls = [url + x['src'] for x in srctaglist]
    
    # make dictionary with lists, pass dictionary to pd.DataFrame()
    mydict = {'Title': titles, 'Price': prices, 'Star Rating': stars, 'Cover Image JPEG URL': imageurls}
    books_df = pd.DataFrame(mydict)
    
    return books_df

In [28]:
# test:
# books_spider('https://books.toscrape.com/catalogue/category/books/philosophy_7/index.html')

### Part h
Notice that there are many pages to http://books.toscrape.com/. When you click on “Next” in the bottom-right corner of the screen, it takes you to http://books.toscrape.com/catalogue/page-2.html. The front page is the same as http://books.toscrape.com/catalogue/page-1.html, and there are 50 total pages.

Write a loop that uses the function you wrote in part g to scrape each of the 50 pages, and append each of these data frames together. If you write this loop correctly, your dataframe will have 1000 rows (20 books on each of the 50 pages). 

Some hints:

* Typing `new_df = pd.DataFrame()` with nothing in the parentheses will create an empty data frame on which new data can be appended.

* There are many loops you can use, but the most straightforward one is a for-values loop that counts from 1 to 50. In Python, you can initialize such a loop with for i in range(1, 51):, and indenting every line below it that belongs inside the loop. Inside the loop, the letter i is now a stand-in for the number currently being considered.

* You will need to figure out how to replace the number in URLs like http://books.toscrape.com/catalogue/page-2.html with the number currently under consideration in the loop. You might need the `str()` function, which turns numeric values into strings.

* `pd.concat()` is a method that appends dataframes together.

[10 points]

In [29]:
pages = 50

new_df = pd.DataFrame() # initialize empty data frame

for i in range(1, pages+1):  # because pages start at 1 and python at 0, adjust range
    url = f'https://books.toscrape.com/catalogue/page-{i}.html' # i used f' strings to replace the number in the URLs
    new_df = pd.concat([new_df, books_spider(url)], ignore_index=True)

new_df


,Title,Price,Star Rating,Cover Image JPEG URL
0,A Light in the Attic,51.77,Three,https://books.toscrape.com/catalogue/page-1.ht...
1,Tipping the Velvet,53.74,One,https://books.toscrape.com/catalogue/page-1.ht...
2,Soumission,50.10,One,https://books.toscrape.com/catalogue/page-1.ht...
3,Sharp Objects,47.82,Four,https://books.toscrape.com/catalogue/page-1.ht...
4,Sapiens: A Brief History of Humankind,54.23,Five,https://books.toscrape.com/catalogue/page-1.ht...
...,...,...,...,...
995,Alice in Wonderland (Alice's Adventures in Won...,55.53,One,https://books.toscrape.com/catalogue/page-50.h...
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",57.06,Four,https://books.toscrape.com/catalogue/page-50.h...
997,A Spy's Devotion (The Regency Spies of London #1),16.97,Five,https://books.toscrape.com/catalogue/page-50.h...
998,1st to Die (Women's Murder Club #1),53.98,One,https://books.toscrape.com/catalogue/page-50.h...
